In [ ]:
import streamlit as st
import pandas as pd
import plotly.express as px

st.set_page_config(
    page_title="Sales Analytics Dashboard",
    layout="wide"
)

st.title("Sales Performance Analytics Dashboard")

df = pd.read_csv("Desktop/DNPROJECT/sales.csv")

df["Order_Date"] = pd.to_datetime(
    df["Order_Date"], errors="coerce"
)

df = df.dropna(subset=["Order_Date", "Sales", "Profit"])

# Sidebar filter
regions = sorted(df["Region"].dropna().unique())

selected_regions = st.sidebar.multiselect(
    "Select Region",
    regions,
    default=regions
)

filtered_df = df[
    df["Region"].isin(selected_regions)
]

# KPI calculations
total_sales = filtered_df["Sales"].sum()
total_profit = filtered_df["Profit"].sum()
total_orders = filtered_df["Order_ID"].nunique()
total_quantity = filtered_df["Quantity"].sum()

profit_margin = (
    total_profit / total_sales * 100
    if total_sales else 0
)

# KPI cards
c1, c2, c3, c4 = st.columns(4)

c1.metric("Total Sales", f"₹{total_sales:,.2f}")
c2.metric("Total Profit", f"₹{total_profit:,.2f}")
c3.metric("Total Orders", f"{total_orders:,}")
c4.metric("Profit Margin", f"{profit_margin:.2f}%")

# Charts
filtered_df = filtered_df.copy()
filtered_df["Month"] = (
    filtered_df["Order_Date"].dt.to_period("M").astype(str)
)

monthly = (
    filtered_df.groupby("Month", as_index=False)["Sales"]
    .sum()
    .sort_values("Month")
)

st.subheader("Monthly Sales Trend")

fig1 = px.line(
    monthly, x="Month", y="Sales", markers=True
)
st.plotly_chart(fig1, use_container_width=True)

category = (
    filtered_df.groupby("Category", as_index=False)["Sales"]
    .sum()
    .sort_values("Sales", ascending=False)
)

st.subheader("Category Performance")

fig2 = px.bar(
    category, x="Category", y="Sales", color="Category"
)
st.plotly_chart(fig2, use_container_width=True)

st.subheader("Sales Data")
st.dataframe(filtered_df, use_container_width=True)